## GRPO Training Algorithm on Llama-3.2-1B-Instruct - Exponential Reward Function

### Environment setup

In [ ]:
# Environment Setup & Repository Cloning

# Change working directory to Kaggle's writable storage
import os
os.chdir('/kaggle/working')

# Remove any previous clone
!rm -rf /kaggle/working/Emergence-of-Thinking

# Clone the official paper repository
!git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git

# Move into the cloned repository
os.chdir('/kaggle/working/Emergence-of-Thinking')

# Fix Ray version from 2.12.0 → 2.31.0 in all config files
print('Fixing Ray version references')
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    for file in files:
        if file.endswith(('.txt', '.py', '.toml')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r') as f:
                    content = f.read()
                if 'ray' in content.lower() and '2.12.0' in content:
                    with open(filepath, 'w') as f:
                        f.write(content.replace('2.12.0', '2.31.0'))
                    print(f'  Fixed: {filepath}')
            except:
                pass


In [ ]:
# Install Dependencies (Kaggle-optimized, with fixes)

!pip install --upgrade pip setuptools wheel --user -q

!pip install torch torchvision torchaudio --user -q

!pip install numpy cython packaging --user -q

# Remove flash-attn from requirements.txt to avoid build failure
print('Removing flash-attn from requirements.txt')
req_file = '/kaggle/working/Emergence-of-Thinking/requirements.txt'
if os.path.exists(req_file):
    with open(req_file, 'r') as f:
        lines = f.readlines()
    with open(req_file, 'w') as f:
        for line in lines:
            if 'flash-attn' not in line.lower():
                f.write(line)
    print('Removed flash-attn from requirements.txt')

# Skip flash-attn build
os.environ['FLASH_ATTENTION_SKIP_CUDA_BUILD'] = 'TRUE'

# Try installing the core project (without flash-attn dependency)
print('Attempting editable install')
!pip install -e . --user -q

# If editable fails, try regular install
if os.system('pip show emergence-of-thinking > /dev/null 2>&1') != 0:
    print('Editable install failed. Trying regular install')
    !pip install . --user -q

# Add repo to Python path
import sys
sys.path.insert(0, '/kaggle/working/Emergence-of-Thinking')
print("Setup complete. Repository added to Python path.")

In [ ]:
!pip install transformers==4.46.3 accelerate --user -q


The following script is used for resumability reasons. More specifically, when a Kaggle session ends or disconnects, the script ensures that if the user uploads the checkpoint files (.csv and .pkl) as a Dataset, they are placed in the correct working directories.

In [ ]:
import os
import shutil

# Dataset path
input_dataset_dir = "/kaggle/input/datasets/kaggle_username/kaggle_dataset"  

working_dir = "/kaggle/working"
sub_output_dir = os.path.join(working_dir, "grpo_length_reward_gradientaccumulationplusnoboxpenalty")


# Create the desiredd subfolders inside /kaggle/working
os.makedirs(sub_output_dir, exist_ok=True)
print("Target subdirectories verified inside /kaggle/working")

# Copy Checkpoint into the root of /kaggle/working
src_checkpoint = os.path.join(input_dataset_dir, "grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl")
dst_checkpoint = os.path.join(working_dir, "grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl")

if os.path.exists(src_checkpoint):
    shutil.copy(src_checkpoint, dst_checkpoint)
    print("Checkpoint perfectly placed at: /kaggle/working/grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl")
else:
    print(f"Looking for: {src_checkpoint}")
    print("Could not find 'grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl' in that specific directory. Check file spelling inside the dataset.")

#  Copy length log into the subfolder
src_log_csv = os.path.join(input_dataset_dir, "length_log.csv")
src_log_raw = os.path.join(input_dataset_dir, "length_log")
dst_log = os.path.join(sub_output_dir, "length_log.csv")

if os.path.exists(src_log_csv):
    shutil.copy(src_log_csv, dst_log)
    print("Metrics log perfectly placed inside subfolder at: /kaggle/working/grpo_length_reward_gradientaccumulationplusnoboxpenalty/length_log.csv")
elif os.path.exists(src_log_raw):
    shutil.copy(src_log_raw, dst_log)
    print("Metrics log (renamed from raw) perfectly placed inside subfolder at: /kaggle/working/grpo_length_reward_gradientaccumulationplusnoboxpenalty/length_log.csv")
else:
    print(f"Could not find 'length_log.csv' or 'length_log' at: {input_dataset_dir}")

#### Model Loading

In order to load the model Llama-3.2-1B-Instruct from Hugging Face, one must have been granted access to the model on Hugging Face and attach the Hugging Face token as a Kaggle Secret.

In [ ]:
# Load Llama 3.2 1B Model with HF Token (Kaggle Secrets)

import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient

# Retrieve HF token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Set cache directories
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/hf_cache'
os.makedirs('/kaggle/working/hf_cache', exist_ok=True)

model_name = "meta-llama/Llama-3.2-1B-Instruct"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)
print("Model loaded successfully!")

# Quick inference test
prompt = "What is 17 + 25? Let's think step by step."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model Response:")
print(response)

In [ ]:
import os
import json

# Verify training data
data_path = '/kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl'
print('Training data exists:', os.path.exists(data_path))

if os.path.exists(data_path):
    with open(data_path, 'r') as f:
        sample = json.loads(f.readline())
    print('Sample keys:', list(sample.keys()))
    print('Sample input (first 200 chars):', sample.get('input', '')[:200])
else:
    print('WARNING: Training data not found. Check the repo structure.')

# Verify evaluation data
eval_data = '/kaggle/working/Emergence-of-Thinking/evaluation/data/math'
print('Eval data exists:', os.path.exists(eval_data))

# Create necessary directories (Kaggle paths)
os.makedirs('/kaggle/working/tmp_code', exist_ok=True)
os.makedirs('/kaggle/working/Emergence-of-Thinking/logs/openrlhf_train_ppo', exist_ok=True)
print('Directories ready.')

In [ ]:
import shutil, os

# Remove previous checkpoint directory if it exists (clean start)
checkpoint_dir = '/kaggle/working/checkpoints'
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir, ignore_errors=True)
    print(f'Removed {checkpoint_dir}')
else:
    print(f'No existing {checkpoint_dir} to remove.')

# Remove Ray temporary directory
ray_tmp = '/tmp/ray'
if os.path.exists(ray_tmp):
    shutil.rmtree(ray_tmp, ignore_errors=True)
    print(f'Removed {ray_tmp}')
else:
    print(f'No existing {ray_tmp} to remove.')

# Check disk usage for Kaggle's working directory
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'/kaggle/working -- Free: {free/1e9:.1f} GB')

# Check /tmp
total, used, free = shutil.disk_usage('/tmp')
print(f'/tmp -- Free: {free/1e9:.1f} GB')

print('Ready for training')

In [ ]:
import os

# Path to deepspeed.py inside the cloned repo
filepath = '/kaggle/working/Emergence-of-Thinking/openrlhf/utils/deepspeed/deepspeed.py'

if os.path.exists(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    patched = False
    new_lines = []
    for line in lines:
        if 'assert state_dict_keys.issubset(' in line and not patched:
            indent = len(line) - len(line.lstrip())
            new_lines.append(' ' * indent + 'state_dict_keys -= {"base_model.model.lm_head.weight"}\n')
            patched = True
        new_lines.append(line)

    if patched:
        with open(filepath, 'w') as f:
            f.writelines(new_lines)
        print('DeepSpeed patched successfully')
    else:
        print('Already patched or pattern not found')
else:
    print(f'File not found: {filepath}')

### GRPO

In [ ]:
!pip install peft datasets latex2sympy2 -q

In [ ]:
%%writefile /kaggle/working/Emergence-of-Thinking/train_grpo_resumable.py
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import json
import numpy as np
import random
import os
import pickle
import argparse
from tqdm import tqdm
import re
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Minimize memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Try loading latex2sympy safely
try:
    from latex2sympy2 import latex2sympy
except ImportError:
    latex2sympy = None

# Reward function
def extract_answer(text):
    match = re.search(r'\\boxed{([^}]+)}', text)
    if match:
        return match.group(1).strip()
    return None

def compute_reward(response, ground_truth, tokenizer, alpha=0.8, C=1.0):
    correct = 0.0
    pred_answer = extract_answer(response)
    if pred_answer is None:
        correct = -0.5
    elif pred_answer is not None and latex2sympy is not None:
        try:
            pred_expr = latex2sympy(pred_answer)
            true_expr = latex2sympy(ground_truth)
            if pred_expr == true_expr:
                correct = 1.0
            else:
                try:
                    pred_float = float(pred_expr)
                    true_float = float(true_expr)
                    if abs(pred_float - true_float) < 1e-6:
                        correct = 1.0
                except (TypeError, ValueError):
                    pass
        except Exception:
            if str(pred_answer).strip() == str(ground_truth).strip():
                correct = 1.0
    else:
        if str(pred_answer).strip() == str(ground_truth).strip():
            correct = 1.0
                
    tokens = tokenizer.encode(response, add_special_tokens=False)
    num_tokens = max(len(tokens), 1)
    exploration_reward = -C * np.exp(-np.sqrt(num_tokens))
    reward = alpha * correct + (1 - alpha) * exploration_reward
    reward = min(10, max(-10,reward))
    return reward, correct

# GRPO training

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.2-1B-Instruct")
    parser.add_argument("--data_path", type=str, default="/kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl")
    parser.add_argument("--output_dir", type=str, default="/kaggle/working/grpo_checkpoints")
    parser.add_argument("--resume_from", type=str, default="/kaggle/working/grpo_checkpoint.pkl")
    parser.add_argument("--max_samples", type=int, default=1000)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--group_size", type=int, default=3)
    parser.add_argument("--max_new_tokens", type=int, default=256)
    parser.add_argument("--learning_rate", type=float, default=1e-5)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--checkpoint_every", type=int, default=20)
    parser.add_argument("--kl_coef", type=float, default=0.01)
    parser.add_argument("--reward_alpha", type=float, default=0.8)
    parser.add_argument("--reward_C", type=float, default=1.0)
    parser.add_argument("--use_lora", action="store_true", default=True)
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=8)
    args = parser.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)

    os.makedirs(args.output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print("Loading Policy Model to GPU 0")
    policy_model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map={"":"cuda:0"},
        attn_implementation="sdpa"
    )
    print(f"Policy Model layer 0 dtype: {policy_model.model.layers[0].self_attn.q_proj.weight.dtype}")
    
    # 4-BIT GRADIENT CHECKPOINTING
    policy_model = prepare_model_for_kbit_training(policy_model)
    if args.use_lora:
        lora_config = LoraConfig(
            r=args.lora_r,
            lora_alpha=args.lora_alpha,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        policy_model = get_peft_model(policy_model, lora_config)
        policy_model.print_trainable_parameters()

    print("Loading Reference Model to GPU 1")
    ref_model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map={"":"cuda:1"},
        attn_implementation="sdpa"
    )
    ref_model.eval()
    for param in ref_model.parameters():
        param.requires_grad = False

    examples = []
    with open(args.data_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= args.max_samples:
                break
            data = json.loads(line)
            examples.append({"prompt": data["input"], "answer": data["answer"]})
    dataset = Dataset.from_list(examples)

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=args.learning_rate)
    
    # Calculate effective steps considering accumulation
    steps_per_epoch = len(dataset) // (args.batch_size * args.gradient_accumulation_steps)
    total_steps = args.epochs * max(1, steps_per_epoch)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    start_step = 0
    start_epoch = 0
    
    if os.path.exists(args.resume_from):
        print(f"Resuming from {args.resume_from}")
        with open(args.resume_from, 'rb') as f:
            checkpoint = pickle.load(f)
        
        policy_model.load_state_dict(checkpoint['policy_state_dict'], strict=False)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_step = checkpoint['step']
        start_epoch = checkpoint.get('epoch', 0)
            
        if 'torch_rng' in checkpoint:
            torch.set_rng_state(checkpoint['torch_rng'])
            np.random.set_state(checkpoint['numpy_rng'])
            random.setstate(checkpoint['random_rng'])
            
        print(f"Resumed successfully. Clean restart from global step {start_step + 1} on Epoch {start_epoch}...")
    else:
        print("Starting fresh training")

    policy_model.train()
    optimizer.zero_grad()

    log_path = os.path.join(args.output_dir, 'length_log.csv')
    if not os.path.exists(log_path):
        with open(log_path, 'w') as f:
            f.write("step,avg_token_length,kl_divergence,accuracy\n")

    for epoch in range(start_epoch, args.epochs):
        indices = list(range(len(dataset)))
        random.seed(args.seed + epoch)
        random.shuffle(indices)
        
        batches = [indices[i:i + args.batch_size] for i in range(0, len(indices), args.batch_size)]
        batches = [b for b in batches if len(b) == args.batch_size]
        
        if epoch == start_epoch:
            completed_batches_in_epoch = start_step % len(batches)
        else:
            completed_batches_in_epoch = 0
            
        active_batches = batches[completed_batches_in_epoch:]
        
        pbar = tqdm(
            active_batches,
            desc=f"Epoch {epoch}",
            initial=completed_batches_in_epoch,
            total=len(batches)
        )

        # Track total expected updates for logging
        total_updates_per_epoch = (len(batches) + args.gradient_accumulation_steps - 1) // args.gradient_accumulation_steps
        total_expected_updates = args.epochs * total_updates_per_epoch
        global_update_step = start_step // args.gradient_accumulation_steps

        # Initialize Cycle Trackers
        cycle_rewards = []
        cycle_correctness = []
        cycle_lengths = []
        cycle_kl_sum = 0.0
        cycle_sequences_count = 0
        accumulated_display_loss = 0.0
        
        for batch_idx, batch_indices in enumerate(pbar):
            step = (epoch * len(batches)) + completed_batches_in_epoch + batch_idx + 1

            prompts = [dataset[i]["prompt"] for i in batch_indices]
            answers = [dataset[i]["answer"] for i in batch_indices]

            all_responses = []
            all_rewards = []
            all_correctness = []
            all_full_tokens = []
            all_prompt_lens = []
            all_token_lengths = []

            for prompt, ans in zip(prompts, answers):
                inputs = tokenizer(prompt, return_tensors="pt").to(policy_model.device)
                input_ids = inputs.input_ids
                attention_mask = inputs.attention_mask
                prompt_len = input_ids.shape[1]
                
                batched_input_ids = input_ids.repeat(args.group_size, 1)
                batched_attention_mask = attention_mask.repeat(args.group_size, 1)
                
                policy_model.eval()
                policy_model.gradient_checkpointing_disable()
                with torch.no_grad():
                    output_ids = policy_model.generate(
                        batched_input_ids,
                        attention_mask=batched_attention_mask,
                        max_new_tokens=args.max_new_tokens,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.95,
                        repetition_penalty=1.15,
                        eos_token_id=[tokenizer.eos_token_id, 128009],
                        pad_token_id=tokenizer.eos_token_id,
                        use_cache=True
                    )
                policy_model.train()
                policy_model.gradient_checkpointing_enable()
                
                group_token_lengths = [] # for tracking
                for sample_idx in range(args.group_size):
                    single_output = output_ids[sample_idx]
                    response_ids = single_output[prompt_len:]
                    response = tokenizer.decode(response_ids, skip_special_tokens=True)
                    
                    reward, is_correct = compute_reward(response, ans, tokenizer, alpha=args.reward_alpha, C=args.reward_C)

                    token_count = len(tokenizer.encode(response, add_special_tokens=False))
                    group_token_lengths.append(token_count)
                    all_responses.append(response)
                    all_rewards.append(reward)
                    all_correctness.append(is_correct)
                    
                    full_seq = torch.cat([input_ids[0], response_ids], dim=-1)
                    all_full_tokens.append(full_seq)
                    all_prompt_lens.append(prompt_len)

                cycle_lengths.extend(group_token_lengths)
            cycle_rewards.extend(all_rewards)
            cycle_correctness.extend(all_correctness)
            
            tqdm.write(f"  > Batch {batch_idx}, Batch Indices {batch_indices} Sample Group Lengths: {group_token_lengths}")
            rewards_tensor = torch.tensor(all_rewards, dtype=torch.float32, device=policy_model.device)
            
            advantages = []
            for i in range(len(prompts)):
                start = i * args.group_size
                end = (i+1) * args.group_size
                group_rewards = rewards_tensor[start:end]
                mean_r = group_rewards.mean()
                std_r = group_rewards.std() + 1e-8
                group_adv = (group_rewards - mean_r) / std_r
                advantages.append(group_adv)
            advantages = torch.cat(advantages)

            del all_responses
            del all_rewards
            torch.cuda.empty_cache()

            loss_fct = torch.nn.CrossEntropyLoss(reduction='none')

            total_sequences = len(all_full_tokens)
            step_kl_sum = 0.0
            
            for idx in range(total_sequences):
                prompt_len = all_prompt_lens[idx]
                
                sub_batch_p = all_full_tokens[idx].unsqueeze(0).to(policy_model.device)
                sub_batch_r = all_full_tokens[idx].unsqueeze(0).to(ref_model.device)

                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    # Forward Policy
                    outputs_p = policy_model(sub_batch_p)
                    logits_p = outputs_p.logits[0, prompt_len-1:-1, :].contiguous()
                    
                    seq_labels_p = sub_batch_p[0, prompt_len:].contiguous()
                    non_pad_mask_p = (seq_labels_p != tokenizer.eos_token_id)
    
                    seq_len = torch.clamp(non_pad_mask_p.sum(), min=1.0)
                    
                    loss_p = loss_fct(logits_p, seq_labels_p)
                    log_prob_p = (-loss_p * non_pad_mask_p).sum()
                    
                    # Forward Reference
                    with torch.no_grad():
                        outputs_r = ref_model(sub_batch_r)
                        logits_r = outputs_r.logits[0, prompt_len-1:-1, :].contiguous()
                    
                    logits_r = logits_r.to(logits_p.device)
                    adv = advantages[idx].to(logits_p.device)
                    
                    log_probs_p = F.log_softmax(logits_p, dim=-1)
                    log_probs_r = F.log_softmax(logits_r, dim=-1)
                    
                    kl_token_wise = (log_probs_p.exp() * (log_probs_p - log_probs_r)).sum(dim=-1)
                    kl_div = (kl_token_wise * non_pad_mask_p).sum()

                    step_kl_sum += kl_div.sum().item()
                    
                    seq_loss = -(adv * log_prob_p - args.kl_coef * kl_div) / total_sequences
                    seq_loss_ga = seq_loss / args.gradient_accumulation_steps
                
                seq_loss_ga.backward()
                accumulated_display_loss += seq_loss.item()
                
                del outputs_p, logits_p, outputs_r, logits_r, loss_p
                if idx % 2 == 0:
                    torch.cuda.empty_cache()

            # Add sequence count and KL to the cycle pile
            cycle_kl_sum += step_kl_sum
            cycle_sequences_count += total_sequences

            # Update weights only when enough gradients have accumulated
            if (batch_idx + 1) % args.gradient_accumulation_steps == 0 or (batch_idx + 1) == len(batches):
                torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad() # Clear for the next accumulation cycle
                
                global_update_step += 1

                # Calculate the true averages for the entire accumulated batch
                true_mean_reward = sum(cycle_rewards) / len(cycle_rewards) if cycle_rewards else 0.0
                true_accuracy = sum(cycle_correctness) / len(cycle_correctness) if cycle_correctness else 0.0
                true_avg_len = sum(cycle_lengths) / len(cycle_lengths) if cycle_lengths else 0.0
                true_avg_kl = cycle_kl_sum / max(cycle_sequences_count, 1)

                # Write everything to the CSV using the global step
                with open(log_path, 'a') as f:
                    f.write(f"{global_update_step},{true_avg_len:.2f},{true_avg_kl:.4f},{true_accuracy:.4f}\n")

                tqdm.write(f"UPDATE {global_update_step}/{total_expected_updates} (Epoch {epoch}) | Loss: {accumulated_display_loss:.4f} | Reward: {true_mean_reward:.4f} | Acc: {true_accuracy:.4f} | Len: {true_avg_len:.1f} | KL: {true_avg_kl:.4f}")

                if global_update_step % args.checkpoint_every == 0:
                    checkpoint = {
                        'policy_state_dict': policy_model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'step': step, # Kept micro-step for accurate dataset resuming
                        'epoch': epoch,
                        'torch_rng': torch.get_rng_state(),
                        'numpy_rng': np.random.get_state(),
                        'random_rng': random.getstate(),
                    }
                    with open(args.resume_from, 'wb') as f:
                        pickle.dump(checkpoint, f)
                    
                    policy_model.save_pretrained(args.output_dir)
                    tokenizer.save_pretrained(args.output_dir)
                    tqdm.write(f"Checkpoint saved and model adapters updated at UPDATE {global_update_step}")

                # Reset trackers for the next cycle
                cycle_rewards = []
                cycle_correctness = []
                cycle_lengths = []
                cycle_kl_sum = 0.0
                cycle_sequences_count = 0
                accumulated_display_loss = 0.0

    policy_model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Training complete. Final model saved to {args.output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ['HF_TOKEN'] = hf_token
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/hf_cache'

!python /kaggle/working/Emergence-of-Thinking/train_grpo_resumable.py \
    --model_name meta-llama/Llama-3.2-1B-Instruct \
    --data_path /kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl \
    --max_samples 5500 \
    --batch_size 2 \
    --group_size 3 \
    --gradient_accumulation_steps 8 \
    --max_new_tokens 2000 \
    --learning_rate 1e-6 \
    --epochs 5 \
    --checkpoint_every 2 \
    --kl_coef 0.05 \
    --reward_alpha 0.8 \
    --reward_C 1000.0 \
    --output_dir /kaggle/working/grpo_length_reward_gradientaccumulationplusnoboxpenalty \
    --resume_from /kaggle/working/grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl